In [ ]:
import numpy as np
import tensorflow as tf
import os
import sys

util_dir = os.getcwd()+ "/utils"
sys.path.append(util_dir)
import Utility
import importlib
importlib.reload(Utility)

model_dir =  os.getcwd()+ "/models/"

In [ ]:
target_layer_name = "add_3"

In [ ]:
import tensorflow as tf
import numpy as np

def compute_gradcam_all_leads(model, x, target_layer_name="add_3"):
    x_tf = tf.cast(x[None, ...], tf.float32)

    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer(target_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, pred = grad_model(x_tf)
        tape.watch(conv_out)
        loss = pred[:, 0]   

    grads = tape.gradient(loss, conv_out) 

    weights = tf.reduce_mean(grads, axis=(1, 2)) 

    cam = tf.reduce_sum(weights[:, None, None, :] * conv_out, axis=-1) 

    cam = tf.nn.relu(cam)[0].numpy()  
    cam = cam / (cam.max() + 1e-8)

    return cam   # shape: (12, T)


In [ ]:

 # 1. Load model
path = model_dir+'ECG_Echo_CNN/'
model_path=path+"/model.h5"
model = tf.keras.models.load_model(model_path)
print("Model loaded:", model_path)
model.summary()    

# 2. load sample
x_test = np.load(path+"x_test.npy")
y_test = np.load(path+"y_test.npy")   
y_pred_test = np.load(path+"y_pred_test.npy")   
print(x_test.shape)


In [ ]:
y_test=y_test.ravel()
y_pred_test= y_pred_test.ravel()
samples = Utility.select_representative_samples(
    x_test,
    y_test,
    y_pred_test,
    percentile=30
)

print(samples["y1"]["idx"])
print(samples["y1"]["y_true"])
print(samples["y1"]["y_pred"])

print(samples["y0"]["idx"])
print(samples["y0"]["y_true"])
print(samples["y0"]["y_pred"])


In [ ]:
x_1 = x_test[samples["y1"]['idx'].ravel().item()]
y_1 = samples["y1"]['y_pred']
cam = compute_gradcam_all_leads(model, x_1)
np.save(path+'cam_1.npy',np.array(cam))
np.save(path+'cam_1_x.npy',np.array(x_1))
np.save(path+'cam_1_y.npy',np.array(y_1))

x_0 = x_test[samples["y0"]['idx'].ravel().item()]
y_0 = samples["y0"]['y_pred']
cam = compute_gradcam_all_leads(model, x_0)
np.save(path+'cam_0.npy',np.array(cam))
np.save(path+'cam_0_x.npy',np.array(x_0))
np.save(path+'cam_0_y.npy',np.array(y_0))